In [ ]:
import requests
import pandas as pd
from datetime import datetime
from urllib.parse import urljoin

#BASE_URL = "https://api.planningcenteronline.com"
#ACCESS_TOKEN = "YOUR_ACCESS_TOKEN_HERE"   # if using OAuth

# If you're using a Personal Access Token instead:
# from requests.auth import HTTPBasicAuth
APP_ID = "ada4e8235441f592b3ec77bd917bae5e2162e20bc08a0a1fadd1ba9632571417"
SECRET = "pco_pat_ecde5fff44868e7dd98f00a4dc98559fe31b60612d0cb802c7ff6d449d1452d4375433f1"

def get(url, params=None):
    """
    Helper for GET requests that follows JSON:API pagination.
    Returns a list of all 'data' items across pages.
    """
    items = []
    while url:
        # Choose one auth style:

        # 1) OAuth / Bearer token:
        #headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
        #r = requests.get(url, headers=headers, params=params)

        # 2) Personal Access Token (comment out Bearer block above and uncomment below):
        r = requests.get(url, auth=HTTPBasicAuth(APP_ID, SECRET), params=params)

        r.raise_for_status()
        payload = r.json()
        items.extend(payload.get("data", []))

        # Follow pagination if there's a "next" link
        links = payload.get("links", {})
        next_link = links.get("next")
        if next_link:
            # next_link is usually absolute; if relative, join with BASE_URL
            url = next_link if next_link.startswith("http") else urljoin(BASE_URL, next_link)
            params = None  # after first page, params are in the URL already
        else:
            url = None
    return items

In [ ]:
def fetch_all_groups():
    url = f"{BASE_URL}/groups/v2/groups"
    return get(url)

def fetch_events_for_group(group_id):
    url = f"{BASE_URL}/groups/v2/groups/{group_id}/events"
    # per_page=100 to reduce calls; adjust as needed
    return get(url + "?per_page=100")

def fetch_attendance_for_event(event_id):
    url = f"{BASE_URL}/groups/v2/events/{event_id}/attendances"
    return get(url + "?per_page=200")  # large enough for typical group sizes

In [ ]:
def build_event_attendance_table():
    groups = fetch_all_groups()
    rows = []

    for g in groups:
        group_id = g["id"]
        group_name = g["attributes"].get("name", f"Group {group_id}")

        events = fetch_events_for_group(group_id)
        for ev in events:
            ev_attrs = ev["attributes"]
            # Skip cancelled events
            status = ev_attrs.get("canceled")
            if status == True:
                continue

            event_id = ev["id"]
            
            # Use starts_at (UTC ISO string); you can switch to starts_at_local if available
            starts_at = ev_attrs.get("starts_at")  # e.g. "2025-10-15T19:00:00Z"
            if not starts_at:
                continue

            # Parse datetime
            event_dt = datetime.fromisoformat(starts_at.replace("Z", "+00:00"))

            attendances = fetch_attendance_for_event(event_id)
            # Count how many attended=True
            attendance_count = sum(
                1 for a in attendances
                if a["attributes"].get("attended") is True
            )

            rows.append({
                "group_id": group_id,
                "group_name": group_name,
                "event_id": event_id,
                "event_datetime": event_dt,
                "attendance_count": attendance_count
            })

    df_events = pd.DataFrame(rows)
    return df_events

df_events = build_event_attendance_table()

# Optional: save raw event-level data
df_events.to_csv("group_event_attendance_raw.csv", index=False)
print(df_events.head())

In [ ]:
# Ensure datetime type
df_events["event_datetime"] = pd.to_datetime(df_events["event_datetime"])

# Compute week start (Monday) for each event
# If you prefer Sunday start, adjust the weekday logic.
# event_datetime is timezone-aware
df_events["week_start"] = (
    df_events["event_datetime"]
    .dt.floor("D")                        # truncate to the day
    .dt.to_pydatetime()                  # convert each timestamp to Python datetime
)

# Now manually move to Monday of that week
df_events["week_start"] = df_events["week_start"].apply(
    lambda d: (d - pd.Timedelta(days=d.weekday()))
)
# Aggregate: total attendance per group per week
df_weekly = (
    df_events
    .groupby(["group_id", "group_name", "week_start"], as_index=False)["attendance_count"]
    .sum()
    .sort_values(["group_name", "week_start"])
)

df_weekly.to_csv("group_weekly_attendance.csv", index=False)
print(df_weekly.head())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# df_plot already cleaned: no future/current weeks, >0 attendance
df_plot["week_start"] = pd.to_datetime(df_plot["week_start"])

for group_name, sub in df_plot.groupby("group_name"):
    sub = sub.sort_values("week_start")

    if sub.empty:
        continue

    # x = evenly spaced index, y = attendance
    x_idx = np.arange(len(sub))          # 0,1,2,...
    y = sub["attendance_count"].to_numpy()
    labels = sub["week_start"].dt.strftime("%Y-%m-%d")

    plt.figure(figsize=(10, 5))

    # points & line
    plt.plot(x_idx, y, marker="o", linestyle="-", label="Attendance")

    # regression on index (even spacing)
    if len(sub) >= 2:
        slope, intercept = np.polyfit(x_idx, y, 1)
        y_hat = slope * x_idx + intercept
        plt.plot(x_idx, y_hat, linestyle="--", label="Trend")

        # convert “per day” slope to “per week”
        slope_per_week = slope
        trend_text = f"{slope_per_week:+.2f} per week"

    # use index as ticks but label with dates
    plt.xticks(ticks=x_idx, labels=labels, rotation=45)

    plt.title(f"Weekly Attendance - {group_name} ({trend_text})")
    plt.xlabel("Week Starting")
    plt.ylabel("Attendance")
    plt.legend()
    plt.tight_layout()
    plt.show()